In [0]:
%sql
SELECT
  statement_id,
  executed_by,
  statement_text,
  query_source.genie_space_id AS genie_space_id,
  client_application,
  execution_status,
  total_duration_ms,
  start_time
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 30 DAYS
  AND query_source.genie_space_id IS NOT NULL
ORDER BY start_time DESC
LIMIT 1000

In [0]:
%sql
SELECT
  query_source.genie_space_id AS genie_space_id,
  executed_by,
  COUNT(*) AS total_queries,
  SUM(CASE WHEN execution_status = 'FINISHED' THEN 1 ELSE 0 END) AS successful_queries,
  SUM(CASE WHEN execution_status = 'FAILED' THEN 1 ELSE 0 END) AS failed_queries,
  ROUND(100.0 * SUM(CASE WHEN execution_status = 'FINISHED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS success_rate_pct,
  ROUND(AVG(total_duration_ms), 0) AS avg_duration_ms,
  ROUND(PERCENTILE(total_duration_ms, 0.5), 0) AS p50_duration_ms,
  ROUND(PERCENTILE(total_duration_ms, 0.95), 0) AS p95_duration_ms,
  MIN(start_time) AS first_query_time,
  MAX(start_time) AS last_query_time
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 30 DAYS
  AND query_source.genie_space_id IS NOT NULL
GROUP BY genie_space_id, executed_by
ORDER BY total_queries DESC

In [0]:
%sql
-- Cost Attribution for a Shared SQL Warehouse
-- Uses total_task_duration_ms as proxy for compute consumption (CPU-time across all cores)

WITH warehouse_usage AS (
  SELECT
    compute.warehouse_id,
    executed_by,
    query_source.genie_space_id,
    client_application,
    COUNT(*) AS query_count,
    SUM(total_duration_ms) AS total_wall_clock_ms,
    COALESCE(SUM(total_task_duration_ms), 0) AS total_task_ms,
    SUM(read_bytes) AS total_read_bytes,
    SUM(written_bytes) AS total_written_bytes,
    SUM(spilled_local_bytes) AS total_spilled_bytes
  FROM system.query.history
  WHERE start_time >= current_date() - INTERVAL 30 DAYS
    AND compute.warehouse_id IS NOT NULL
    -- Optionally filter to a specific warehouse:
    -- AND compute.warehouse_id = '<your_warehouse_id>'
  GROUP BY compute.warehouse_id, executed_by, query_source.genie_space_id, client_application
),
warehouse_totals AS (
  SELECT
    warehouse_id,
    SUM(total_task_ms) AS warehouse_total_task_ms
  FROM warehouse_usage
  GROUP BY warehouse_id
)
SELECT
  u.warehouse_id,
  u.executed_by,
  u.client_application,
  u.genie_space_id,
  u.query_count,
  u.total_task_ms,
  ROUND(try_divide(100.0 * u.total_task_ms, t.warehouse_total_task_ms), 2) AS pct_of_warehouse_cost,
  ROUND(u.total_task_ms / 1000.0 / 3600.0, 2) AS task_hours,
  ROUND(try_divide(u.total_read_bytes, 1073741824), 2) AS read_gb,
  ROUND(try_divide(u.total_wall_clock_ms, u.query_count), 0) AS avg_query_duration_ms
FROM warehouse_usage u
JOIN warehouse_totals t ON u.warehouse_id = t.warehouse_id
WHERE t.warehouse_total_task_ms > 0
ORDER BY u.warehouse_id, pct_of_warehouse_cost DESC

In [0]:
%sql
    
-- DBU + Dollar Cost Attribution by User per Warehouse
-- Joins query history → billing usage → list prices for actual $ cost

WITH user_share AS (
  SELECT
    compute.warehouse_id,
    executed_by,
    SUM(total_task_duration_ms) AS user_task_ms,
    SUM(SUM(total_task_duration_ms)) OVER (PARTITION BY compute.warehouse_id) AS warehouse_total_task_ms
  FROM system.query.history
  WHERE start_time >= current_date() - INTERVAL 30 DAYS
    AND compute.warehouse_id IS NOT NULL
  GROUP BY compute.warehouse_id, executed_by
),
warehouse_dbus AS (
  SELECT
    u.usage_metadata.warehouse_id AS warehouse_id,
    u.sku_name,
    SUM(u.usage_quantity) AS total_dbus,
    SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)) AS total_cost_usd
  FROM system.billing.usage u
  LEFT JOIN system.billing.list_prices lp
    ON u.sku_name = lp.sku_name
    AND u.usage_unit = lp.usage_unit
    AND u.cloud = lp.cloud
    AND lp.price_start_time <= u.usage_start_time
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_start_time)
  WHERE u.usage_start_time >= current_date() - INTERVAL 30 DAYS
    AND u.billing_origin_product = 'SQL'
    AND u.usage_metadata.warehouse_id IS NOT NULL
  GROUP BY u.usage_metadata.warehouse_id, u.sku_name
)
SELECT
  s.warehouse_id,
  s.executed_by,
  ROUND(100.0 * s.user_task_ms / s.warehouse_total_task_ms, 2) AS pct_share,
  d.sku_name,
  d.total_dbus AS warehouse_total_dbus,
  ROUND(d.total_dbus * (s.user_task_ms / s.warehouse_total_task_ms), 2) AS attributed_dbus,
  ROUND(d.total_cost_usd, 2) AS warehouse_total_cost_usd,
  ROUND(d.total_cost_usd * (s.user_task_ms / s.warehouse_total_task_ms), 2) AS attributed_cost_usd
FROM user_share s
LEFT JOIN warehouse_dbus d ON s.warehouse_id = d.warehouse_id
WHERE s.warehouse_total_task_ms > 0
ORDER BY attributed_cost_usd DESC